## 0. Tiêu đề & Thông tin nhóm

# Đề tài: Khử nhiễu ảnh dựa trên ảnh thật

**Môn:** Xử lý ảnh & Thị giác máy tính  
**Nhóm:** N04

| STT | MSSV | Họ và tên | Vai trò |
|-----|------|-----------|--------|
| 1   |      |           |        |
| 2   |      |           |        |
| 3   |      |           |        |
| 4   |      |           |        |

> **TODO:** Điền đầy đủ thông tin thành viên nhóm vào bảng trên.

## 1. Setup

Import thư viện, cố định seed, khai báo đường dẫn dữ liệu và hàm tiện ích dùng chung.

> **TODO:** Thêm các thư viện bổ sung nếu cần. Đảm bảo seed cố định để kết quả tái lập.

In [ ]:
# === 1.1 Import ===
import os
import time
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from pathlib import Path

# Seed cố định
SEED = 42
np.random.seed(SEED)

# Đường dẫn
DEV_DIR = Path("data/dev")
EVAL_DIR = Path("data/eval")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Setup OK")

### 1.1 Hàm tiện ích

Các hàm hiển thị ảnh, tính PSNR, SSIM, và đo thời gian.

> **TODO:** Cài đặt nội dung bên trong mỗi hàm.

In [ ]:
def show_images(images, titles=None, figsize=(15, 5), cmap="gray"):
    """Hiển thị danh sách ảnh trên cùng một hàng.

    Parameters
    ----------
    images : list[np.ndarray]
        Danh sách ảnh cần hiển thị.
    titles : list[str] | None
        Tiêu đề từng ảnh.
    figsize : tuple
        Kích thước figure.
    cmap : str
        Colormap (dùng cho ảnh xám).
    """
    # TODO: Cài đặt hiển thị bằng matplotlib
    raise NotImplementedError


def psnr(clean, denoised, data_range=255):
    """Tính Peak Signal-to-Noise Ratio.

    Parameters
    ----------
    clean : np.ndarray
        Ảnh sạch tham chiếu.
    denoised : np.ndarray
        Ảnh sau khử nhiễu.
    data_range : int
        Giá trị pixel tối đa.

    Returns
    -------
    float
        Giá trị PSNR (dB).
    """
    # TODO: Dùng skimage.metrics.peak_signal_noise_ratio hoặc tự tính
    raise NotImplementedError


def ssim(clean, denoised, data_range=255):
    """Tính Structural Similarity Index.

    Parameters
    ----------
    clean : np.ndarray
        Ảnh sạch tham chiếu.
    denoised : np.ndarray
        Ảnh sau khử nhiễu.
    data_range : int
        Giá trị pixel tối đa.

    Returns
    -------
    float
        Giá trị SSIM.
    """
    # TODO: Dùng skimage.metrics.structural_similarity
    raise NotImplementedError


def timeit(func, *args, n_runs=3, **kwargs):
    """Đo thời gian trung bình chạy hàm.

    Parameters
    ----------
    func : callable
        Hàm cần đo.
    *args, **kwargs
        Tham số truyền cho func.
    n_runs : int
        Số lần chạy lấy trung bình.

    Returns
    -------
    result : Any
        Kết quả trả về từ func.
    elapsed : float
        Thời gian trung bình (giây).
    """
    # TODO: Cài đặt đo thời gian
    raise NotImplementedError

## 2. Tuyên bố thách thức

Phần này trình bày bối cảnh bài toán, thách thức, phương pháp dự kiến và dự đoán **TRƯỚC** khi chạy thí nghiệm.

### 2A. Bối cảnh, tác vụ, dữ liệu, thách thức C, hạn chế F, giả định thu hẹp

> **TODO:** Viết các nội dung sau:
> - **Bối cảnh:** Nhiễu cảm biến khi chụp ảnh thật (ISO cao, thiếu sáng) ảnh hưởng chất lượng ảnh.
> - **Tác vụ:** Khử nhiễu ảnh (image denoising) nhằm khôi phục ảnh sạch từ ảnh nhiễu.
> - **Dữ liệu:** Mô tả nguồn dữ liệu ảnh thật (ví dụ: SIDD, hoặc tự chụp).
> - **Thách thức C:** Nhiễu cảm biến trên ảnh thật — phức tạp hơn nhiễu Gaussian tổng hợp.
> - **Hạn chế F:** Gaussian filter muốn khử nhiễu mạnh phải tăng σ → cạnh & chi tiết mịn bị nhòe.
> - **Giả định thu hẹp:** Liệt kê các giả định đơn giản hóa (ví dụ: nhiễu gần Gaussian, ảnh tĩnh, v.v.).

### 2B. Kỹ thuật P, cơ chế hoạt động, dự đoán điểm gãy

> **TODO:** Viết các nội dung sau (TRƯỚC khi chạy bất kỳ thí nghiệm nào):
> - **Kỹ thuật P:** Bilateral filter với σ_r cố định, chọn trên tập dev.
> - **Cơ chế:** Giải thích tại sao bilateral filter bảo toàn cạnh (range kernel chặn lan truyền qua biên).
> - **Biến thể:** Immerkær ước lượng σ nhiễu → bilateral với σ_r = k·σ̂, tự thích nghi theo ảnh.
> - **Dự đoán điểm gãy:** Dự đoán phương pháp P/biến thể sẽ thất bại khi nào
>   (ví dụ: σ nhiễu ≈ ΔI cạnh; nhiễu phụ thuộc tín hiệu; nén JPEG; ảnh nhiều texture).

### 2C. Thước đo và Baseline

> **TODO:** Viết các nội dung sau:
> - **Thước đo:** PSNR (đo sai số pixel), SSIM (đo cấu trúc cảm nhận), thời gian xử lý.
> - **Baseline:** Gaussian filter — lọc tuyến tính đẳng hướng, không phân biệt cạnh/vùng phẳng.
> - Giải thích tại sao baseline là điểm so sánh công bằng.

## 3. Dữ liệu & Phân tầng

Tải dữ liệu ảnh thật (cặp noisy-clean), ước lượng σ thật, phân tầng cường độ nhiễu.

> **TODO:** Mô tả nguồn dữ liệu cụ thể: bộ dữ liệu công khai (SIDD, DND, ...) hoặc tự chụp.
> Giải thích cách có ảnh sạch tham chiếu (ISO thấp + tripod, trung bình burst, hoặc ground truth có sẵn).

In [ ]:
def load_pairs(data_dir):
    """Tải các cặp ảnh (noisy, clean) từ thư mục.

    Parameters
    ----------
    data_dir : str | Path
        Thư mục chứa ảnh, cấu trúc:
          data_dir/noisy/  — ảnh nhiễu
          data_dir/clean/  — ảnh sạch tương ứng

    Returns
    -------
    list[dict]
        Mỗi phần tử: {'name': str, 'noisy': np.ndarray, 'clean': np.ndarray}
    """
    # TODO: Đọc ảnh, ghép cặp theo tên file
    raise NotImplementedError


def estimate_true_sigma(noisy, clean):
    """Ước lượng σ nhiễu thật từ cặp ảnh.

    σ_true = std(noisy − clean)

    Parameters
    ----------
    noisy : np.ndarray
        Ảnh nhiễu.
    clean : np.ndarray
        Ảnh sạch tham chiếu.

    Returns
    -------
    float
        Giá trị σ thật.
    """
    # TODO: Tính std(noisy - clean)
    raise NotImplementedError


def stratify(pairs, thresholds=(10, 25)):
    """Phân tầng cặp ảnh theo cường độ nhiễu.

    Tầng:
      - 'light'   : σ_true < thresholds[0]
      - 'moderate' : thresholds[0] ≤ σ_true < thresholds[1]
      - 'heavy'   : σ_true ≥ thresholds[1]

    Parameters
    ----------
    pairs : list[dict]
        Danh sách cặp ảnh từ load_pairs.
    thresholds : tuple[float, float]
        Ngưỡng phân tầng.

    Returns
    -------
    pd.DataFrame
        Cột: name, sigma_true, stratum
    """
    # TODO: Tính σ thật cho mỗi cặp, gán tầng, trả DataFrame
    raise NotImplementedError

In [ ]:
# === 3.1 Tải dữ liệu & phân tầng ===

# TODO: Gọi load_pairs, stratify; lưu strata.csv
# dev_pairs = load_pairs(DEV_DIR)
# eval_pairs = load_pairs(EVAL_DIR)
# strata_df = stratify(eval_pairs)
# strata_df.to_csv(OUTPUT_DIR / "strata.csv", index=False)
# strata_df
pass

### 3.1 Hiển thị ảnh mẫu mỗi tầng

> **TODO:** Chọn 1–2 ảnh đại diện mỗi tầng (nhẹ / trung bình / nặng), hiển thị cặp noisy-clean cạnh nhau. Nhận xét trực quan về mức nhiễu.

In [ ]:
# TODO: Hiển thị ảnh mẫu mỗi tầng
# Gợi ý: lọc strata_df theo stratum, lấy ảnh, gọi show_images
pass

## 4. Cài đặt phương pháp

Ba phương pháp khử nhiễu: Baseline (Gaussian), P (Bilateral cố định), Biến thể (Bilateral thích nghi qua Immerkær).

### 4.1 Baseline — Gaussian Filter

Lọc Gaussian là bộ lọc tuyến tính đẳng hướng: mỗi pixel được thay bằng trung bình có trọng số Gaussian của láng giềng. Trọng số chỉ phụ thuộc khoảng cách không gian, **không** phụ thuộc cường độ → không phân biệt cạnh và vùng phẳng.

> **TODO:** Giải thích thêm công thức, ưu/nhược điểm.

In [ ]:
def denoise_gaussian(img, ksize=5, sigma=1.0):
    """Khử nhiễu bằng Gaussian filter.

    Parameters
    ----------
    img : np.ndarray
        Ảnh đầu vào (uint8 hoặc float).
    ksize : int
        Kích thước kernel (số lẻ).
    sigma : float
        Độ lệch chuẩn không gian của Gaussian.

    Returns
    -------
    np.ndarray
        Ảnh sau lọc.
    """
    # TODO: Dùng cv2.GaussianBlur
    raise NotImplementedError

### 4.2 Phương pháp P — Bilateral Filter (σ_r cố định)

Bilateral filter kết hợp **domain kernel** (không gian, như Gaussian) và **range kernel** (cường độ). Range kernel giảm trọng số pixel có cường độ khác xa pixel trung tâm → bảo toàn cạnh.

Tham số σ_r cố định, chọn trên tập dev.

> **TODO:** Giải thích công thức bilateral, vai trò σ_s và σ_r, tại sao bảo toàn cạnh.

In [ ]:
def denoise_bilateral(img, d=9, sigma_s=75, sigma_r=75):
    """Khử nhiễu bằng Bilateral filter.

    Parameters
    ----------
    img : np.ndarray
        Ảnh đầu vào.
    d : int
        Đường kính vùng lân cận.
    sigma_s : float
        σ không gian (spatial).
    sigma_r : float
        σ cường độ (range), cố định.

    Returns
    -------
    np.ndarray
        Ảnh sau lọc.
    """
    # TODO: Dùng cv2.bilateralFilter
    raise NotImplementedError

### 4.3 Ước lượng σ nhiễu — Immerkær

Phương pháp Immerkær ước lượng σ nhiễu từ **một ảnh duy nhất** (không cần ảnh sạch) bằng Laplacian convolution + robust MAD estimator.

> **TODO:** Trình bày công thức Immerkær, giải thích tại sao dùng Laplacian, ưu/nhược điểm.

In [ ]:
def immerkaer_sigma(img):
    """Ước lượng σ nhiễu theo phương pháp Immerkær.

    Dùng Laplacian kernel 3×3 và robust estimator.

    Parameters
    ----------
    img : np.ndarray
        Ảnh đầu vào (grayscale, uint8 hoặc float).

    Returns
    -------
    float
        σ̂ — ước lượng σ nhiễu.
    """
    # TODO: Cài đặt Immerkær (Laplacian conv → MAD → σ̂)
    raise NotImplementedError

### 4.4 Biến thể — Bilateral thích nghi (Immerkær + Bilateral)

Kết hợp Immerkær để ước lượng σ̂ → đặt σ_r = k·σ̂. Nhờ đó σ_r tự thích nghi theo mức nhiễu từng ảnh, thay vì cố định.

> **TODO:** Giải thích tại sao thích nghi tốt hơn cố định; hệ số k chọn trên dev.

In [ ]:
def denoise_adaptive(img, d=9, sigma_s=75, k=1.0):
    """Khử nhiễu bằng Bilateral filter thích nghi.

    σ_r = k * immerkaer_sigma(img)

    Parameters
    ----------
    img : np.ndarray
        Ảnh đầu vào.
    d : int
        Đường kính vùng lân cận.
    sigma_s : float
        σ không gian.
    k : float
        Hệ số nhân σ̂ để tạo σ_r.

    Returns
    -------
    np.ndarray
        Ảnh sau lọc.
    """
    # TODO: Gọi immerkaer_sigma → tính sigma_r → gọi denoise_bilateral
    raise NotImplementedError

### 4.5 Kiểm tra độ chính xác Immerkær

So sánh σ̂ (ước lượng Immerkær) với σ thật (= std(noisy − clean)) trên tập eval.

> **TODO:** Tạo scatter plot σ̂ vs σ_true, tính tương quan Pearson, nhận xét sai lệch.

In [ ]:
# TODO: Với mỗi cặp ảnh eval:
#   - σ̂ = immerkaer_sigma(noisy)
#   - σ_true = estimate_true_sigma(noisy, clean)
# Vẽ scatter plot σ̂ vs σ_true + đường y=x
# Tính Pearson correlation
pass

## 5. E1 — Chứng cứ thách thức

Chạy baseline (Gaussian filter) trên tập eval theo từng tầng, cho thấy baseline thất bại ở nhiễu trung bình/nặng.

> **TODO:**
> - Chạy baseline trên toàn bộ eval, tính PSNR/SSIM theo tầng.
> - Lập bảng chỉ số trung bình mỗi tầng.
> - Chọn ≥ 3 ảnh thất bại tiêu biểu, hiển thị (noisy → denoised → clean).
> - Viết lập luận nguyên nhân thất bại liên quan đến nhiễu (C) và hạn chế (F).

In [ ]:
# === E1: Chạy baseline Gaussian theo tầng ===

# TODO: Chạy denoise_gaussian trên eval_pairs, nhóm theo tầng
# Tính PSNR, SSIM cho từng ảnh; tổng hợp trung bình mỗi tầng
# Tạo DataFrame kết quả
pass

In [ ]:
# === E1: Hiển thị ≥ 3 ảnh thất bại tiêu biểu ===

# TODO: Chọn ảnh có PSNR thấp nhất hoặc SSIM thấp nhất
# Hiển thị: noisy | denoised (Gaussian) | clean
pass

### Nhận xét E1

> **TODO:** Lập luận:
> - Baseline thất bại chủ yếu ở tầng nào?
> - Nguyên nhân: nhiễu mạnh → tăng σ Gaussian → mất cạnh/chi tiết (F).
> - Kết luận: cần phương pháp bảo toàn cạnh.

## 6. E2 — Khảo sát tham số

Khảo sát ảnh hưởng của từng tham số lên chất lượng khử nhiễu. **Chỉ dùng tập dev.**

> **TODO:** Mỗi tham số thử ≥ 3 giá trị. Hiển thị ảnh + chỉ số song song. Kèm ảnh trung gian (noisy, σ̂, output, residual).

### 6.1 Gaussian: khảo sát ksize và σ

> **TODO:** Thử ≥ 3 ksize (ví dụ 3, 5, 7, 9) và ≥ 3 σ (ví dụ 0.5, 1, 2, 4). Hiển thị kết quả, nhận xét.

In [ ]:
# === E2: Khảo sát Gaussian ===

# TODO: Vòng lặp ksize × sigma
# Gọi denoise_gaussian, tính PSNR/SSIM
# Hiển thị ảnh output + residual
# Lưu kết quả vào DataFrame
pass

### 6.2 Bilateral: khảo sát d, σ_s, σ_r

> **TODO:** Cố định 2 tham số, thay đổi 1 tham số (≥ 3 giá trị). Hiển thị kết quả, nhận xét.

In [ ]:
# === E2: Khảo sát Bilateral ===

# TODO: Khảo sát từng tham số d, sigma_s, sigma_r
# Gọi denoise_bilateral, tính PSNR/SSIM
# Hiển thị ảnh output + residual
pass

### 6.3 Biến thể: khảo sát hệ số k

> **TODO:** Thử ≥ 3 giá trị k (ví dụ 0.5, 1.0, 1.5, 2.0). Hiển thị ảnh + σ̂ + output + residual.

In [ ]:
# === E2: Khảo sát hệ số k ===

# TODO: Vòng lặp k
# Gọi denoise_adaptive, tính PSNR/SSIM
# Hiển thị ảnh, in σ̂ ước lượng
pass

### 6.4 Hiển thị ảnh trung gian

> **TODO:** Chọn 1 ảnh dev đại diện, hiển thị pipeline đầy đủ:
> ảnh nhiễu → σ̂ (in giá trị) → output (mỗi phương pháp) → residual (noisy − output).
> Nhận xét theo lý thuyết.

In [ ]:
# === E2: Ảnh trung gian ===

# TODO: Chọn 1 ảnh, hiển thị pipeline
# Noisy | Gaussian output | Bilateral output | Adaptive output
# Residual maps tương ứng
pass

### Nhận xét E2

> **TODO:** Nhận xét kết quả khảo sát tham số:
> - Gaussian: σ lớn → PSNR giảm do mất chi tiết?
> - Bilateral: σ_r quá nhỏ → không khử đủ, quá lớn → mất cạnh?
> - Hệ số k: giá trị tối ưu nằm ở khoảng nào?
> - Residual: residual lý tưởng ≈ nhiễu thuần (không có cấu trúc ảnh).

## 7. So sánh định lượng 3 phương pháp theo tầng

Chạy 3 phương pháp (Gaussian, Bilateral cố định, Bilateral thích nghi) trên **tập eval**, so sánh PSNR, SSIM, thời gian theo từng tầng.

> **TODO:** Dùng tham số tối ưu chọn từ E2 (trên dev). Tạo bảng + biểu đồ đường.

In [ ]:
# === So sánh 3 phương pháp trên eval ===

# TODO: Tham số tối ưu (chọn từ E2 trên dev)
# BEST_GAUSS = {'ksize': ..., 'sigma': ...}
# BEST_BILATERAL = {'d': ..., 'sigma_s': ..., 'sigma_r': ...}
# BEST_ADAPTIVE = {'d': ..., 'sigma_s': ..., 'k': ...}

# TODO: Chạy 3 phương pháp trên mỗi ảnh eval
# Tính PSNR, SSIM, thời gian
# Nhóm theo tầng, tính trung bình
pass

In [ ]:
# === Bảng so sánh ===

# TODO: Tạo bảng: hàng = tầng, cột = (PSNR, SSIM, time) × 3 phương pháp
# Hiển thị bằng pd.DataFrame
pass

In [ ]:
# === Biểu đồ đường PSNR/SSIM theo tầng ===

# TODO: Biểu đồ đường: trục x = tầng (nhẹ, trung bình, nặng)
# 3 đường (Gaussian, Bilateral, Adaptive)
# Subplot 1: PSNR, Subplot 2: SSIM, Subplot 3: Thời gian
pass

### Nhận xét so sánh

> **TODO:** Nhận xét:
> - Phương pháp nào tốt nhất ở mỗi tầng?
> - Bilateral thích nghi có vượt Bilateral cố định không? Ở tầng nào rõ nhất?
> - Thời gian xử lý: Bilateral chậm hơn Gaussian bao nhiêu?
> - Kết quả có phù hợp dự đoán ở mục 2B không?

## 8. E3 — Ablation

Phân tích vai trò từng thành phần bằng cách tắt lần lượt:

- **(a)** Tắt range kernel: đặt σ_r → rất lớn (10⁶) → Bilateral ≈ Gaussian.
- **(b)** Tắt Immerkær: dùng σ_r cố định (= P) thay vì thích nghi.

> **TODO:** Chạy trên eval, lập bảng so sánh, nhận xét.

In [ ]:
# === E3a: Tắt range kernel ===

# TODO: Gọi denoise_bilateral với sigma_r = 1e6
# So sánh PSNR/SSIM với Bilateral bình thường và Gaussian
pass

In [ ]:
# === E3b: Tắt Immerkær ===

# TODO: So sánh denoise_bilateral (σ_r cố định) vs denoise_adaptive (σ_r thích nghi)
# Chạy trên eval, nhóm theo tầng
pass

In [ ]:
# === E3: Bảng tổng hợp ablation ===

# TODO: Tạo bảng:
# | Cấu hình             | PSNR (nhẹ) | PSNR (TB) | PSNR (nặng) | SSIM ... |
# | Full (Adaptive)      |            |           |             |          |
# | − Range kernel (a)   |            |           |             |          |
# | − Immerkær (b)       |            |           |             |          |
pass

### Nhận xét E3

> **TODO:** Nhận xét:
> - (a) Khi tắt range kernel, PSNR/SSIM giảm bao nhiêu so với Bilateral đầy đủ? → Khẳng định vai trò bảo toàn cạnh.
> - (b) Khi tắt Immerkær, chênh lệch lớn nhất ở tầng nào? → Immerkær giúp ở ảnh nhiễu nặng hay nhẹ?
> - Thành phần nào đóng góp quan trọng nhất?

## 9. E4 — Điểm gãy

Thêm nhiễu Gaussian có kiểm soát (σ_add tăng dần) lên ảnh thật, quan sát khi nào phương pháp P/biến thể bắt đầu thất bại.

> **TODO:**
> - Chọn 1–2 ảnh eval sạch nhất (tầng nhẹ).
> - Thêm nhiễu σ_add = 5, 10, 15, ..., 60.
> - Chạy 3 phương pháp, vẽ biểu đồ PSNR/SSIM theo σ_add.
> - Xác định ngưỡng gãy: σ_add mà tại đó PSNR/SSIM giảm mạnh hoặc Bilateral ≈ Gaussian.
> - Đối chiếu dự đoán (mục 2B).

In [ ]:
# === E4: Thêm nhiễu có kiểm soát ===

# TODO:
# sigma_adds = [5, 10, 15, 20, 25, 30, 40, 50, 60]
# Với mỗi σ_add:
#   noisy_synth = clean + N(0, σ_add)
#   Chạy 3 phương pháp, tính PSNR/SSIM
pass

In [ ]:
# === E4: Biểu đồ PSNR/SSIM theo σ_add ===

# TODO: Biểu đồ đường
# Trục x: σ_add, trục y: PSNR (hoặc SSIM)
# 3 đường: Gaussian, Bilateral, Adaptive
# Đánh dấu ngưỡng gãy
pass

### Nhận xét E4

> **TODO:** Nhận xét và đối chiếu dự đoán:
> - Ngưỡng gãy tìm được: σ_add ≈ ?
> - So với dự đoán: σ nhiễu ≈ ΔI (gradient cạnh) → bilateral không phân biệt nhiễu/cạnh.
> - Immerkær có lệch khi: nhiễu phụ thuộc tín hiệu (Poisson), nén JPEG, ảnh nhiều texture.
> - Dự đoán đúng/sai/đúng một phần?

## 10. Thảo luận & Kết luận

> **TODO:** Viết thảo luận tổng hợp:
> - **Dự đoán đúng:** Những dự đoán nào ở mục 2B được thí nghiệm xác nhận? Cụ thể hoá bằng số liệu.
> - **Dự đoán sai:** Những dự đoán nào bị bác bỏ? Vì sao?
> - **Đúng một phần:** Dự đoán nào đúng hướng nhưng cần điều chỉnh?
> - **Giả định yếu nhất:** Giả định nào ở mục 2A dễ bị vi phạm nhất trên dữ liệu thật? Hậu quả?
> - **Hạn chế & hướng phát triển:** Non-local means, BM3D, deep learning, v.v.
> - **Kết luận:** Tóm tắt đóng góp chính.

## 11. Xuất kết quả

Lưu bảng CSV và hình ảnh vào `outputs/` để dùng cho báo cáo.

> **TODO:** Đảm bảo mọi bảng số liệu và hình quan trọng đều được lưu.

In [ ]:
# === Xuất CSV ===

# TODO: Lưu các DataFrame kết quả
# - strata.csv (đã lưu ở mục 3)
# - e1_baseline_results.csv
# - e2_param_search.csv
# - comparison_by_stratum.csv
# - e3_ablation.csv
# - e4_breakpoint.csv

# csv_files = {
#     'e1_baseline_results.csv': ...,
#     'comparison_by_stratum.csv': ...,
#     'e3_ablation.csv': ...,
#     'e4_breakpoint.csv': ...,
# }
# for fname, df in csv_files.items():
#     df.to_csv(OUTPUT_DIR / fname, index=False)
#     print(f"Saved {fname}")
pass

In [ ]:
# === Xuất hình ===

# TODO: Lưu các figure quan trọng
# Gợi ý: dùng fig.savefig(OUTPUT_DIR / "ten_hinh.png", dpi=150, bbox_inches='tight')
# - comparison_psnr_by_stratum.png
# - comparison_ssim_by_stratum.png
# - e4_breakpoint_psnr.png
# - immerkaer_scatter.png
# - failure_examples.png
pass

In [ ]:
print("=== Hoàn tất! Kết quả đã lưu tại:", OUTPUT_DIR.resolve(), "===")